# SYJ Video Forge GPU worker — v0.3.0 PRE-RELEASE

**UNVERIFIED ON REAL HARDWARE.** This notebook is a user-operated GPU worker. It reads a manually supplied `job.json`, runs a selected model locally in this notebook session, and writes `result.mp4` plus `manifest.json`. It does not create a tunnel, start an HTTP service, or automatically bundle/download output files.

## Kaggle

Enable Internet in notebook settings if permitted and choose an available GPU accelerator. Free GPU availability and session limits are not guaranteed; some sessions may offer roughly 16 GB-class devices such as a P100 or T4, but compatibility is unverified. Check current platform terms and model license before use.

## Google Colab

Choose Runtime → Change runtime type → GPU if available. Free GPU access, session lifetime, and resource limits are not guaranteed. Review the current Colab FAQ and terms; deepfakes and certain remote/web-service uses are prohibited. Do not use this workflow for prohibited content.

## Privacy

The job bundle includes the prompt and optional starting image. Upload it only to a notebook or machine you choose. The notebook does not upload job contents to any other destination than the model download provider and the GPU session you selected.


In [ ]:
import os, sys, platform, subprocess, shutil, json, time, urllib.request
from pathlib import Path
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM bytes:", torch.cuda.get_device_properties(0).total_memory)
        print("Compute capability:", torch.cuda.get_device_capability(0))
        print("Native bf16 reported:", getattr(torch.cuda, "is_bf16_supported", lambda: False)())
except Exception as exc:
    print("PyTorch/GPU inspection failed:", repr(exc))
try:
    with urllib.request.urlopen("https://huggingface.co", timeout=10) as response:
        print("Internet check: HTTP", response.status)
except Exception as exc:
    raise RuntimeError("Internet check failed. Enable Internet if permitted by your notebook platform and retry.") from exc
print("Free disk bytes:", shutil.disk_usage(Path.cwd()).free)


## Fetch the pinned worker source

This notebook expects the public `v0.3.0` tag after it has been published. If you are testing an untagged overlay, use your reviewed branch/commit instead and record it in the verification report. The clone command does not request any credentials.

In [ ]:
REPO = Path.cwd() / "SYJ-Video-Forge"
if not (REPO / "worker" / "cli.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "v0.3.0", "https://github.com/SHalimoosavi/SYJ-Video-Forge.git", str(REPO)], check=True)
else:
    print("Using existing repository:", REPO)
sys.path.insert(0, str(REPO))
print("Repository:", REPO)


## Install the pinned worker dependencies

PyTorch is intentionally not listed in `requirements-worker.txt`; keep the notebook image's CUDA-compatible PyTorch when possible. If the installed PyTorch/CUDA combination is incompatible, resolve that in the platform environment before proceeding. These pins have not been installed or tested on a GPU in the authoring environment.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-worker.txt")], check=True)


## Optional Hugging Face authentication

The selected model pages were not marked gated in the research snapshot. If the exact model page requires access, first review and accept its license on Hugging Face. If your platform provides a secret named `HF_TOKEN`, it is read from the environment. Alternatively, set `SYJ_HF_TOKEN_REQUIRED=1` and enter a token at the hidden prompt below. Never paste a token into a notebook cell or commit it.

In [ ]:
if os.environ.get("SYJ_HF_TOKEN_REQUIRED") == "1" and not os.environ.get("HF_TOKEN"):
    from getpass import getpass
    token = getpass("Hugging Face token (hidden input; optional): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
print("HF_TOKEN configured:", bool(os.environ.get("HF_TOKEN")))


## Smoke test

This downloads model weights as needed, loads the selected engine, generates one short low-resolution clip, checks for degenerate frames, records elapsed time and hardware/library information, and writes `worker-output/smoke.json`. The command exits non-zero on failure. Real GPU behavior remains **UNVERIFIED ON REAL HARDWARE** until this cell is run by the user and the report is reviewed.

In [ ]:
OUT = Path.cwd() / "worker-output"
OUT.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "worker", "smoke-test", "--engine", "ltx-video", "--out", str(OUT)], cwd=REPO, check=True)
print((OUT / "smoke.json").read_text(encoding="utf-8"))


## Supply `job.json`

Use a bundle downloaded from the SYJ Video Forge browser UI. On Kaggle, attach it as a dataset/input file if that fits your privacy requirements. On Colab, upload the file using the widget below. If neither platform upload helper is available, enter a local path. Do not upload private prompts/images unless you trust the selected platform.

In [ ]:
job_path = None
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    candidates = list(kaggle_input.rglob("job.json"))
    if candidates:
        print("Found candidate bundles:")
        for i, item in enumerate(candidates): print(i, item)
        choice = input("Choose candidate index [0]: ").strip() or "0"
        job_path = candidates[int(choice)]
if job_path is None:
    try:
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        if uploaded:
            name = next((n for n in uploaded if n.lower().endswith(".json")), None)
            if name:
                target = Path.cwd() / Path(name).name
                target.write_bytes(uploaded[name])
                job_path = target
    except Exception:
        pass
if job_path is None:
    raw = input("Path to job.json: ").strip()
    job_path = Path(raw).expanduser()
job = json.loads(Path(job_path).read_text(encoding="utf-8"))
if job.get("schemaVersion") != 1:
    raise ValueError("Expected schemaVersion 1 job bundle")
engine_id = job.get("engine", {}).get("id", "ltx-video")
print("Job ID:", job.get("jobId"), "Engine:", engine_id, "Scenes:", len(job.get("scenes", [])))


## Run the job with progress output

The worker writes each scene as it finishes and updates `manifest.json` incrementally. If the session is interrupted, rerun the same command with `--resume`; only scene files whose SHA-256 still matches the manifest are skipped.

In [ ]:
JOB_OUT = Path.cwd() / "worker-output" / "job-result"
cmd = [sys.executable, "-m", "worker", "run", str(Path(job_path).resolve()), "--out", str(JOB_OUT), "--engine", engine_id]
print("Command:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
if code != 0:
    raise RuntimeError(f"Worker failed with exit code {code}; inspect the printed error and manifest.")


## Inspect and download the two result files

No ZIP is created automatically. The files are listed below:


In [ ]:
for name in ("result.mp4", "manifest.json"):
    p = JOB_OUT / name
    print(f"{name}: {p} — exists={p.is_file()} size_bytes={p.stat().st_size if p.is_file() else 'missing'}")
if not (JOB_OUT / "result.mp4").is_file() or not (JOB_OUT / "manifest.json").is_file():
    raise FileNotFoundError("Expected result.mp4 and manifest.json are missing")
print("Kaggle: download each file from the notebook Output/Files panel (or save the output files as a notebook version).")
print("Colab: use the Files sidebar to download result.mp4 and manifest.json separately, or run files.download on each file individually.")
print("Do not change the manifest's SHA-256 or claim AI generation if the worker failed. Hardware/model fields are self-reported.")
